Build an exploratory learned component-selector pilot recovered from PR244.

CPU HistGBM fitting is performed in the generated notebook after GPU proposal
collection. A/B are filename partitions of detector training data, not independent
base-model OOF or guaranteed group-disjoint samples. Held-out results were already
exposed historically; independent confirmation is required for scientific use.
Selector training labels use overlap with any annotator as an exploratory target.
PQ instead scores the same predictions separately per annotator and pools counts;
the union training label is not a claim of an official matched true positive.

Pre-provision dependencies and artifacts. Independently freeze
artifacts/v64_expected_contract_foldN.json before execution. Its schema extends
the v62 input contract with runtime_versions, config_sha256 and selector_split_sha256. The latter
hashes UTF-8 sorted JSON {train_a:[...],train_b:[...],heldout:[...]} using the
deterministic split below. preprocessed_sha256 covers ALL consumed filenames.
protocol_sha256 binds this builder, the v62 builder and embedded model source.
Full keys: schema_version=1, fold, protocol_sha256, detector_sha256,
refiner_sha256, annotations_sha256, folds_sha256, preprocessed_sha256,
config_sha256, selector_split_sha256, runtime_versions, baseline.
config_sha256 hashes UTF-8 json.dumps(mapping, sort_keys=True) for CFG keys
YOLO_IMGSZ, CROP_SIZE, CROP_CONTEXT, CONF, PREDICT_CONF_FLOOR, PREDICT_IOU,
REFINER_THRESHOLD, DROP_THRESHOLD, MIN_AREA, preserving their int/float types.
baseline={pq,tolerance} must be separately frozen for the matching protocol.
No canonical identity or tolerance is manufactured by this builder.

No cached proposals, classifier, grid, or partial manifest are reused. Only a
complete baseline-gated exploratory manifest is published atomically. Publication
reserves a sibling .publish-lock directory exclusively and checks the result path
inside that reservation. Concurrent publishers and cell reruns cannot replace a
completed result. A reservation left by a killed runtime fails closed: preserve
it and select a new output path rather than automatically reclaiming it.
Linux publication requires an unambiguous local filesystem; stacked mount points and
FUSE/network mount targets are refused without relying on mount listing order.
The default manifest lives under /content/solar_v64_selector. Download the
completed file before ending the Colab runtime; Drive durability is not assumed.
The actual Colab filesystem/runtime remains unverified. A preflight runs before
input staging/model loading, with another check at publication after evaluation.
Preinstall the listed runtime packages before execution; dependency versions are
checked before staging. All training files are staged; local capacity and copy
time have not been measured in Colab.
Generation
alone performs no inference, fitting, downloads or external execution.


Runtime identity keys: python, numpy, pandas, scipy, torch, opencv-python, ultralytics, pycocotools, scikit-learn. Restart runtime and rerun all cells after interruption.

In [ ]:
# [1/6] Drive接続、設定、前提の確認
import hashlib, json, math, random, shutil, sys, time, warnings
from datetime import datetime, timezone
from importlib.metadata import version
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

class CFG:
    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v64_selector")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    FOLD = 2
    PROTOCOL_SHA256 = 'ea6165084bb23a08d623753bb68aca6a8c75b95b3e8828f9c34b36a2981185ba'
    # Shared production postprocessing; exploratory selector discovery
    YOLO_IMGSZ = 1280
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    CONF = 0.26
    PREDICT_CONF_FLOOR = 0.05
    PREDICT_IOU = 0.55
    REFINER_THRESHOLD = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    # Exploratory selector output and frozen input contract
    MANIFEST = LOCAL_ROOT / "v64_selector_fold2_diagnostic_v1_manifest.json"
    EXPECTED_CONTRACT = ARTIFACT_ROOT / f"v64_expected_contract_fold{FOLD}.json"

def prod_checkpoint(fold):
    return CFG.DRIVE_ROOT / "runs" / f"yolo_fold_{fold}" / "weights" / "best.pt"

for path in [CFG.LOCAL_ROOT, CFG.PREPROC_DIR]:
    path.mkdir(parents=True, exist_ok=True)

required = {
    "competition data": CFG.DATA_CACHE,
    "grouped_folds.csv": CFG.ARTIFACT_ROOT / "grouped_folds.csv",
    "前処理画像キャッシュ": CFG.PREPROC_CACHE,
    f"fold{CFG.FOLD} prod検出器": prod_checkpoint(CFG.FOLD),
    f"fold{CFG.FOLD} 標準refiner": CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"前提artifactが見つかりません: {missing}")
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({k: "OK" for k in required})
print({"GPU": torch.cuda.get_device_name(0), "torch": torch.__version__})

def file_sha256(path):
    import hashlib
    from pathlib import Path
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def verified_checkpoint_snapshot(source, expected_sha256, destination):
    # A unique local copy isolates the loader from subsequent Drive syncs.
    # Verify the bytes actually copied, including any replacement during copy.
    import shutil
    import tempfile
    from pathlib import Path
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destination, suffix=".pt", delete=False) as handle:
        local = Path(handle.name)
    try:
        shutil.copyfile(source, local)
        if file_sha256(local) != expected_sha256:
            raise ValueError("Copied checkpoint differs from frozen input identity")
        return local
    except BaseException:
        local.unlink(missing_ok=True)
        raise


def validate_fold_rows(rows, files, fold):
    from pathlib import PurePosixPath
    if type(fold) is not int or fold not in (0, 1, 2):
        raise ValueError("Invalid evaluation fold")
    mapping = {}
    for row in rows:
        name, value = row.get("file_name"), row.get("fold")
        if (not isinstance(name, str) or not name or "\\" in name
                or PurePosixPath(name).name != name or ":" in name or name in (".", "..")
                or type(value) is not int or value not in (0, 1, 2) or name in mapping):
            raise ValueError("Invalid or duplicate fold assignment")
        mapping[name] = value
    if set(mapping) != set(files):
        raise ValueError("Fold assignments must cover exactly the consumed files")
    if not any(v == fold for v in mapping.values()) or not any(v != fold for v in mapping.values()):
        raise ValueError("Training and validation partitions must be nonempty")
    return mapping


def validate_baseline(measured, expected, tolerance):
    import math
    if (any(type(v) not in (int, float) or not math.isfinite(v) for v in (measured, expected, tolerance))
            or not 0 <= measured <= 1 or not 0 <= expected <= 1 or not 0 <= tolerance < 1
            or abs(measured - expected) > tolerance):
        raise ValueError("Baseline reproduction guard failed; no result accepted")


def require_expected_contract(path, actual):
    import json
    import re
    from pathlib import Path
    expected = json.loads(Path(path).read_text(encoding="utf-8"))
    keys = {"schema_version", "fold", "protocol_sha256", "detector_sha256", "refiner_sha256",
            "annotations_sha256", "folds_sha256", "preprocessed_sha256", "baseline"}
    if not isinstance(expected, dict) or set(expected) != keys:
        raise ValueError("Expected frozen input contract is incomplete")
    baseline = expected["baseline"]
    if not isinstance(baseline, dict) or set(baseline) != {"pq", "tolerance"}:
        raise ValueError("Explicit baseline reproduction guard is required")
    validate_baseline(baseline["pq"], baseline["pq"], baseline["tolerance"])
    if type(expected["schema_version"]) is not int or type(expected["fold"]) is not int:
        raise ValueError("Invalid contract schema/fold")
    for key in keys - {"schema_version", "fold", "baseline"}:
        if not isinstance(expected[key], str) or re.fullmatch("[0-9a-f]{64}", expected[key]) is None:
            raise ValueError("Invalid expected SHA256")
    if {k:v for k,v in expected.items() if k != "baseline"} != actual:
        raise ValueError("Frozen input contract differs; do not infer")
    return expected


def validate_existing_manifest(path, expected):
    import json
    from pathlib import Path
    path = Path(path)
    if path.exists():
        manifest = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(manifest, dict) or manifest.get("input_contract") != expected:
            raise ValueError("Existing manifest contract differs; preserve it and choose a new path")


def atomic_json(path, payload):
    import json
    import os
    import tempfile
    from pathlib import Path
    path = Path(path)
    data = json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False).encode("utf-8")
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode="wb", dir=path.parent, suffix=".tmp", delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def require_local_publication(path, mountinfo=None):
    import re
    import sys
    from pathlib import Path
    path = Path(path).resolve()
    if mountinfo is None:
        if sys.platform == "win32":
            if path.drive.startswith("\\\\"):
                raise ValueError("Manifest publication requires a local filesystem")
            return  # CPU-test support only; mapped Windows drives are not classified.
        mountinfo = Path("/proc/self/mountinfo").read_text(encoding="utf-8")
    matches = []
    for line in mountinfo.splitlines():
        before, separator, after = line.partition(" - ")
        fields = before.split()
        if not separator or len(fields) < 5 or not after.split():
            continue
        point = Path(re.sub(r"\\([0-7]{3})", lambda m: chr(int(m[1], 8)), fields[4])).resolve()
        if point == path or point in path.parents:
            matches.append((len(point.parts), after.split()[0]))
    deepest = max((depth for depth, _ in matches), default=-1)
    candidates = [kind for depth, kind in matches if depth == deepest]
    if len(candidates) != 1 or candidates[0] not in {
            "overlay", "ext2", "ext3", "ext4", "xfs", "btrfs", "tmpfs", "ramfs", "rootfs", "zfs"}:
        raise ValueError("Manifest publication requires a verified local filesystem")


def publish_manifest(path, payload):
    from pathlib import Path
    path = Path(path)
    require_local_publication(path)
    reservation = path.with_name(path.name + ".publish-lock")
    try:
        reservation.mkdir()
    except FileExistsError as error:
        raise ValueError("Manifest publication is reserved; choose a new output path") from error
    try:
        if path.exists():
            raise ValueError("Manifest already exists; preserve it and choose a new output path")
        atomic_json(path, payload)
    finally:
        reservation.rmdir()


def selector_partitions(mapping, fold):
    import random
    training = sorted(n for n, f in mapping.items() if f != fold)
    heldout = sorted(n for n, f in mapping.items() if f == fold)
    rng = random.Random(3407)
    training = sorted(training, key=lambda name: rng.random())
    middle = len(training) // 2
    a, b = training[:middle], training[middle:]
    if not a or not b or not heldout:
        raise ValueError("Selector partitions must all be nonempty")
    return a, b, heldout


def fit_selector(x, y):
    import numpy as np
    from sklearn.ensemble import HistGradientBoostingClassifier
    if (x.ndim != 2 or x.shape[1] != 10 or len(x) != len(y)
            or not np.isfinite(x).all() or set(np.unique(y)) != {False, True}):
        raise ValueError("Finite training features and two classes are required")
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=.08,
                                          max_depth=6, min_samples_leaf=20, random_state=3407)
    model.fit(x, y)
    return model


def require_selector_contract(path, actual):
    import json
    import re
    from pathlib import Path
    expected = json.loads(Path(path).read_bytes())
    if not isinstance(expected, dict) or set(expected) != set(actual) | {"baseline"}:
        raise ValueError("Incomplete independently frozen selector contract")
    baseline = expected["baseline"]
    if not isinstance(baseline, dict) or set(baseline) != {"pq", "tolerance"}:
        raise ValueError("Explicit frozen baseline is required")
    # Schema/range validation only; EVALUATE compares the measured baseline.
    validate_baseline(baseline["pq"], baseline["pq"], baseline["tolerance"])
    if type(expected["schema_version"]) is not int or type(expected["fold"]) is not int:
        raise ValueError("Invalid schema/fold")
    for key in actual:
        if key.endswith("sha256") and (not isinstance(expected[key], str)
                or re.fullmatch("[0-9a-f]{64}", expected[key]) is None):
            raise ValueError("Invalid SHA256 identity")
    if {k: v for k, v in expected.items() if k != "baseline"} != actual:
        raise ValueError("Frozen selector inputs differ; do not load models")
    return expected


def validate_rows(rows, names):
    import numpy as np
    if len(rows) != len(names) or {r["name"] for r in rows} != set(names):
        raise ValueError("Missing or duplicate proposal rows")
    for row in rows:
        for p in row["props"]:
            if (set(p["feat"]) != set(FEATURE_KEYS)
                    or not np.isfinite([p["feat"][k] for k in FEATURE_KEYS]).all()
                    or not 0 <= p["conf"] <= 1 or type(p["label_hit"]) is not bool):
                raise ValueError("Invalid proposal features/label")


def validate_metrics(metrics):
    import math
    for key in ("tp", "fp", "fn"):
        if type(metrics[key]) is not int or metrics[key] < 0:
            raise ValueError("Invalid official counts")
    tp, fp, fn = (metrics[k] for k in ("tp", "fp", "fn"))
    denominator = tp + .5 * fp + .5 * fn
    if metrics["denominator"] != denominator:
        raise ValueError("Invalid official denominator")
    if any(not math.isfinite(metrics[k]) or not 0 <= metrics[k] <= 1
           for k in ("pq", "sq", "rq")):
        raise ValueError("Invalid metric range")
    rq = tp / denominator if denominator else 0.0
    if (not math.isclose(metrics["rq"], rq, abs_tol=1e-12)
            or not math.isclose(metrics["pq"], metrics["sq"] * rq, abs_tol=1e-12)
            or (tp == 0 and metrics["sq"] != 0)):
        raise ValueError("Inconsistent official metrics")
    return metrics


def decode_verified_image(captured, expected_hash):
    import hashlib
    import cv2
    import numpy as np
    if hashlib.sha256(captured).hexdigest() != expected_hash:
        raise ValueError("Image bytes changed after contract verification")
    image = cv2.imdecode(np.frombuffer(captured, np.uint8), cv2.IMREAD_COLOR)
    if image is None or image.shape[:2] != (2048, 2048):
        raise ValueError("Production protocol requires decodable 2048x2048 images")
    return image


In [ ]:
require_local_publication(CFG.MANIFEST)
# Dependencies must be preinstalled; refuse missing packages before any staging.
RUNTIME_VERSIONS = {"python": sys.version.split()[0], **{
    package: version(package) for package in ("numpy", "pandas", "scipy", "torch",
        "opencv-python", "ultralytics", "pycocotools", "scikit-learn")}}
# Stage all selector inputs and verify the frozen contract.
def find_dataset_root(base):
    matches = list(Path(base).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if not matches:
        raise FileNotFoundError(f"学習annotation JSONが見つかりません: {base}")
    if len(matches) != 1:
        raise ValueError("Exactly one annotation JSON must be selected")
    return matches[0].parents[2]

TRAIN_JSON = next(find_dataset_root(CFG.DATA_CACHE).rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
# Capture each remote metadata file once: parse and fingerprint the same bytes.
annotation_bytes = TRAIN_JSON.read_bytes()
coco = json.loads(annotation_bytes.decode("utf-8"))

images = {item["id"]: item for item in coco["images"]}
if len(images) != len(coco["images"]):
    raise ValueError("Duplicate annotation image IDs")
annotations_by_image = defaultdict(list)
for ann in coco["annotations"]:
    if ann["image_id"] not in images:
        raise ValueError("Annotation references an unknown image")
    annotations_by_image[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

from io import BytesIO
fold_bytes = (CFG.ARTIFACT_ROOT / "grouped_folds.csv").read_bytes()
fold_frame = pd.read_csv(BytesIO(fold_bytes))
fold_of_file = validate_fold_rows(fold_frame.to_dict("records"), train_files, CFG.FOLD)
train_files_fold = [n for n in train_files if fold_of_file[n] != CFG.FOLD]
valid_files_fold = [n for n in train_files if fold_of_file[n] == CFG.FOLD]

for name in tqdm(train_files, desc="前処理画像をSSDへ"):
    local = CFG.PREPROC_DIR / name
    shutil.copy2(CFG.PREPROC_CACHE / name, local)  # replace stale local cache

PREPROC_HASHES = {name: file_sha256(CFG.PREPROC_DIR / name) for name in sorted(train_files)}
ACTUAL_INPUTS = {
    "schema_version": 1, "fold": CFG.FOLD, "protocol_sha256": CFG.PROTOCOL_SHA256,
    "detector_sha256": file_sha256(prod_checkpoint(CFG.FOLD)),
    "refiner_sha256": file_sha256(CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt"),
    "annotations_sha256": hashlib.sha256(annotation_bytes).hexdigest(),
    "folds_sha256": hashlib.sha256(fold_bytes).hexdigest(),
    "preprocessed_sha256": hashlib.sha256(json.dumps(PREPROC_HASHES, sort_keys=True).encode("utf-8")).hexdigest(),
}
train_A_files, train_B_files, valid_files_fold = selector_partitions(fold_of_file, CFG.FOLD)
PARTITIONS = {"train_a": train_A_files, "train_b": train_B_files, "heldout": valid_files_fold}
ACTUAL_INPUTS["selector_split_sha256"] = hashlib.sha256(json.dumps(PARTITIONS, sort_keys=True).encode()).hexdigest()
ACTUAL_INPUTS["config_sha256"] = hashlib.sha256(json.dumps({
    key: getattr(CFG, key) for key in ("YOLO_IMGSZ", "CROP_SIZE", "CROP_CONTEXT", "CONF",
        "PREDICT_CONF_FLOOR", "PREDICT_IOU", "REFINER_THRESHOLD", "DROP_THRESHOLD", "MIN_AREA")
}, sort_keys=True).encode()).hexdigest()
ACTUAL_INPUTS["runtime_versions"] = RUNTIME_VERSIONS
if CFG.MANIFEST.exists():
    raise ValueError("Complete manifest already exists; preserve it and choose a new output path")
INPUT_CONTRACT = require_selector_contract(CFG.EXPECTED_CONTRACT, ACTUAL_INPUTS)
validate_existing_manifest(CFG.MANIFEST, INPUT_CONTRACT)
print({"input_contract": "verified before model loading", "protocol_sha256": CFG.PROTOCOL_SHA256})



In [ ]:
started = time.time()
started_utc = datetime.now(timezone.utc).isoformat()
torch.cuda.reset_peak_memory_stats()
# [3/6] MediSegUNet(solar_filament/models/mediseg_models.py と同一)+ refinerロード
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)

refiner = MediSegUNet(3, 1, False).cuda().eval()
_verified_refiner = verified_checkpoint_snapshot(
    CFG.ARTIFACT_ROOT / f"mediseg_refiner_fold{CFG.FOLD}_resume.pt",
    INPUT_CONTRACT["refiner_sha256"], CFG.LOCAL_ROOT / "verified_checkpoints")
_ckpt = torch.load(_verified_refiner,
                    map_location="cuda", weights_only=False)
refiner.load_state_dict(_ckpt.get("best_model") or _ckpt["model"], strict=True)
del _ckpt


In [ ]:
# [4/6] 推論・後処理・採点関数(build_v61系と同一経路+refiner内部統計の取得)
from pycocotools import mask as mask_utils
from ultralytics import YOLO

def result_masks(result):
    if result.masks is None:
        return [], np.zeros(0, np.float32)
    height, width = result.orig_shape
    output = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        output.append(mask >= 0.5)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32) if result.boxes is not None else np.zeros(0, np.float32)
    return output, confidences

def mask_square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

@torch.no_grad()
def refine_proposal_stats(image, proposal):
    """確定構成のshrink maskと、verifier特徴用のrefiner prob統計を返す。"""
    x0, y0, x1, y1 = mask_square_box(proposal, context=CFG.CROP_CONTEXT)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
    prob_full = np.zeros(proposal.shape, dtype=np.float32)
    prob_full[y0:y1, x0:x1] = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    refined = (prob_full >= CFG.REFINER_THRESHOLD) & proposal
    inside = prob_full[proposal]
    probmass = float(inside.mean()) if inside.size else 0.0
    return refined, probmass

def encode_one(mask):
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))

def decode_one(rle):
    return mask_utils.decode(rle) != 0

def greedy_disjoint(masks, order, min_area=CFG.MIN_AREA):
    if not masks:
        return []
    occupied = np.zeros(masks[0].shape, dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices(masks)
    return greedy_disjoint(masks, sorted(subset, key=lambda i: areas[i]))

def to_rle(annotation, height, width):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return mask_utils.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

def mask_iou(a, b):
    inter = float((a & b).sum())
    union = float((a | b).sum())
    return inter / union if union > 0 else 0.0

class PanopticQuality:
    def __init__(self, iou_threshold=0.5):
        self.iou_threshold = iou_threshold
        self.tp_ious = []
        self.fp_count = 0
        self.fn_count = 0
    def add_matrix(self, matrix, n_gt, n_pred):
        if n_gt == 0:
            self.fp_count += n_pred; return
        if n_pred == 0:
            self.fn_count += n_gt; return
        hits = matrix > self.iou_threshold
        scores = matrix[hits].tolist()
        self.tp_ious.extend(scores)
        self.fp_count += int((hits.sum(axis=0) == 0).sum())
        self.fn_count += int((hits.sum(axis=1) == 0).sum())
    @property
    def denominator(self):
        return len(self.tp_ious) + 0.5 * self.fp_count + 0.5 * self.fn_count
    @property
    def score(self):
        d = self.denominator
        return float(sum(self.tp_ious) / d) if d > 0 else 0.0
    def sq_rq(self):
        tp = len(self.tp_ious)
        sq = float(sum(self.tp_ious) / tp) if tp > 0 else 0.0
        rq = float(tp / (tp + 0.5 * self.fp_count + 0.5 * self.fn_count)) if (tp + self.fp_count + self.fn_count) > 0 else 0.0
        return sq, rq
    def summary(self):
        sq, rq = self.sq_rq()
        return {"pq": self.score, "sq": sq, "rq": rq, "tp": len(self.tp_ious),
                "fp": self.fp_count, "fn": self.fn_count, "denominator": self.denominator}

gt_rles = {}
for image_id, info in images.items():
    height, width = int(info["height"]), int(info["width"])
    gt_rles[image_id] = [to_rle(ann, height, width) for ann in annotations_by_image[image_id] if not ann.get("iscrowd", 0)]

print("後処理・採点関数の定義OK")

FEATURE_KEYS = ["conf", "probmass", "ref_raw_iou", "raw_area", "ref_area",
                "area_ratio", "aspect", "solidity", "darkness", "limb"]

def proposal_features(image, proposal, conf, refined, probmass):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.float32)
    ys, xs = np.where(proposal)
    w, h = xs.max() - xs.min() + 1, ys.max() - ys.min() + 1
    contours, _ = cv2.findContours(proposal.astype(np.uint8), cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    hull_area = sum(cv2.contourArea(cv2.convexHull(c)) for c in contours) or 1.0
    ring = (cv2.dilate(proposal.astype(np.uint8),
                      cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))) > 0) & ~proposal
    inside = float(gray[proposal].mean())
    darkness = (float(gray[ring].mean()) if ring.any() else inside) - inside
    disk = gray > max(20.0, float(gray.max()) * .15)
    if disk.sum() > 1000:
        dys, dxs = np.where(disk)
        cx, cy, radius = dxs.mean(), dys.mean(), np.sqrt(disk.sum() / np.pi)
    else:
        cy, cx = np.array(proposal.shape) / 2
        radius = min(proposal.shape) * 900 / 2048
    raw_area, ref_area = float(proposal.sum()), float(refined.sum())
    return dict(zip(FEATURE_KEYS, [float(conf), float(probmass), mask_iou(refined, proposal),
        raw_area, ref_area, ref_area / raw_area, float(max(w, h) / min(w, h)),
        raw_area / hull_area, darkness,
        float(np.hypot(xs.mean() - cx, ys.mean() - cy) / max(radius, 1.0))]))


In [ ]:
# Collect fresh in-memory RLE proposals; no legacy/partial cache reuse.
verified_detector = verified_checkpoint_snapshot(prod_checkpoint(CFG.FOLD),
    INPUT_CONTRACT["detector_sha256"], CFG.LOCAL_ROOT / "verified_checkpoints")
yolo_model = YOLO(str(verified_detector))

def collect(names, label):
    rows = []
    for name in tqdm(names, desc=label):
        captured = (CFG.PREPROC_DIR / name).read_bytes()
        image = decode_verified_image(captured, PREPROC_HASHES[name])
        if any((images[i]["height"], images[i]["width"]) != image.shape[:2]
               for i in file_to_ids[name]):
            raise ValueError("Annotation/image dimensions differ")
        result = yolo_model.predict(image.copy(), imgsz=CFG.YOLO_IMGSZ,
            conf=CFG.PREDICT_CONF_FLOOR, iou=CFG.PREDICT_IOU,
            agnostic_nms=True, retina_masks=True, verbose=False)[0]
        masks, confidences = result_masks(result)
        if len(masks) != len(confidences):
            raise ValueError("Detector mask/confidence count differs")
        props = []
        for mask, conf in zip(masks, confidences):
            if mask.sum() < CFG.MIN_AREA:
                continue
            refined, probmass = refine_proposal_stats(image, mask)
            props.append({"conf": float(conf), "refined_rle": encode_one(refined),
                          "feat": proposal_features(image, mask, conf, refined, probmass)})
        # Union-annotator overlap is a training target, not a matched-TP claim.
        truth = [r for image_id in file_to_ids[name] for r in gt_rles[image_id]]
        matrix = iou_matrix(truth, [p["refined_rle"] for p in props])
        for j, p in enumerate(props):
            p["label_hit"] = bool(len(truth) and matrix[:, j].max() > .5)
        rows.append({"name": name, "props": props})
    validate_rows(rows, names)
    return rows

trainA_rows = collect(train_A_files, "train-A discovery proposals")
trainB_rows = collect(train_B_files, "train-B discovery proposals")
val_rows = collect(valid_files_fold, "historically exposed heldout proposals")
del yolo_model
torch.cuda.empty_cache()


In [ ]:
def feature_matrix(rows):
    return np.asarray([[p["feat"][k] for k in FEATURE_KEYS]
        for row in rows for p in row["props"]], dtype=float).reshape(-1, len(FEATURE_KEYS))

def score_rows(rows, conf_floor, probabilities=None, p_keep=None):
    if probabilities is not None and (len(probabilities) != sum(len(r["props"]) for r in rows)
            or not np.isfinite(probabilities).all()
            or ((probabilities < 0) | (probabilities > 1)).any()):
        raise ValueError("Classifier probability coverage/range differs")
    metric = PanopticQuality()
    offset = 0
    for row in rows:
        keep = []
        for p in row["props"]:
            probability = probabilities[offset] if probabilities is not None else 1.0
            offset += 1
            if p["conf"] >= conf_floor and (p_keep is None or probability >= p_keep):
                keep.append(p)
        final = apply_ownership([decode_one(p["refined_rle"]) for p in keep],
                                [p["conf"] for p in keep])
        prediction = [encode_one(mask) for mask in final]
        for image_id in file_to_ids[row["name"]]:
            truth = gt_rles[image_id]
            metric.add_matrix(iou_matrix(truth, prediction), len(truth), len(prediction))
    return validate_metrics(metric.summary())

# Reproduce separately frozen baseline before selector diagnostics or fitting.
baseline = score_rows(val_rows, CFG.CONF)
validate_baseline(baseline["pq"], INPUT_CONTRACT["baseline"]["pq"],
                  INPUT_CONTRACT["baseline"]["tolerance"])
x_a = feature_matrix(trainA_rows)
y_a = np.asarray([p["label_hit"] for row in trainA_rows for p in row["props"]])
selector = fit_selector(x_a, y_a)
x_b = feature_matrix(trainB_rows)
if not len(x_b):
    raise ValueError("Train-B has no proposals; operation point cannot be selected")
prob_b = selector.predict_proba(x_b)[:, 1]
grid = []
for conf_floor in tqdm([.05, .08, .12, .20, .26, .35], desc="Train-B operation grid"):
    for p_keep in [0, .3, .5, .7, .85, .93]:
        metrics = score_rows(trainB_rows, conf_floor, prob_b, p_keep)
        grid.append({"conf_floor": conf_floor, "p_keep": p_keep, "metrics": metrics})
# Deterministic ties prefer the earliest predeclared point. No heldout grid.
chosen = max(grid, key=lambda item: item["metrics"]["pq"])
x_val = feature_matrix(val_rows)
prob_val = selector.predict_proba(x_val)[:, 1] if len(x_val) else np.zeros(0)
selected = score_rows(val_rows, chosen["conf_floor"], prob_val, chosen["p_keep"])
manifest = {
    "schema_version": 1, "kind": "exploratory_v64_selector_complete",
    "scientific_acceptance": "unverified; historically exposed heldout; independent confirmation required",
    "input_contract": INPUT_CONTRACT, "partitions": PARTITIONS,
    "feature_keys": FEATURE_KEYS, "union_annotator_overlap_target": True,
    "proposal_counts": {"train_a": len(x_a), "train_b": len(x_b), "heldout": len(x_val)},
    "train_b_grid": grid, "frozen_operation_point": chosen,
    "baseline": baseline, "selected": selected, "delta_pq": selected["pq"] - baseline["pq"],
    "runtime": {"service": "Google Colab", "gpu": torch.cuda.get_device_name(0),
        "started_at_utc": started_utc, "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "scope": "refiner loading, proposal collection, CPU selector fitting and scoring; excludes input staging",
        "gpu_peak_vram_bytes": int(torch.cuda.max_memory_allocated()),
        "elapsed_seconds": time.time() - started, "compute_units_used": None,
        "compute_units_note": "Record separately from Colab UI; runtime API unavailable"},
}
publish_manifest(CFG.MANIFEST, manifest)
print(json.dumps({"baseline": baseline, "selected": selected, "manifest": str(CFG.MANIFEST)}))


Download the completed local manifest from the Colab Files pane before ending the runtime. Its path is printed above. Do not redirect MANIFEST to Drive/FUSE: publication rejects nonlocal mounts. A copied file is a backup; its durability is not validated here.